# Inference and resolution

_Artificial Intelligence (CS221)_

**Mechanical rules that crank out new true facts from old ones.**

Checking truth tables is slow when there are many symbols.

     Inference rules derive new facts directly, by pattern. No table needed.

---

This notebook is a practice scaffold for the **Inference and resolution** lesson. We'll build a tiny **forward-chaining** engine — the workhorse behind modus ponens — and watch it grind out new facts one rule-firing at a time. Run each cell, read the note above it, then try the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python

We'll implement **forward chaining** over a set of *definite clauses* (Horn clauses). Each rule has the form `premises ⇒ conclusion`: if every premise is already known to be true, **modus ponens** lets us conclude the conclusion. We build it in three steps: (1) write the rules and starting facts, (2) fire rules until nothing new appears, (3) read off what got proven.

### Step 1 — Write the rules and the starting facts

A rule is a pair `(premises, conclusion)`: a list of symbols that must all hold, and the single symbol we may then derive. This is a **Horn clause** — at most one positive conclusion — which is exactly the shape forward chaining knows how to handle.

We seed the knowledge base with the facts we already accept as true. Here we only know `A`; everything else must be *earned* by applying a rule.

In [ ]:
# Each rule is (premises, conclusion). Definite clauses (Horn): all premises ⇒ one conclusion.
rules = [
    (["A", "B"], "C"),     # if A and B, then C
    (["C"], "D"),          # if C, then D
    (["A"], "B"),          # if A, then B
    (["D", "C"], "E"),     # if D and C, then E
]

facts = {"A"}              # the only thing we know to start

print("starting facts:", sorted(facts))

### Step 2 — Fire rules until nothing new appears

Forward chaining is a fixed-point loop. On each sweep we test every rule: if its conclusion is not yet known **and** all of its premises are already facts, the rule *fires* and we add the conclusion to our knowledge base. That single addition is **modus ponens** in action.

We keep sweeping as long as any rule fired during the pass. Once a full sweep adds nothing, we've reached the *closure* — every fact that can be derived has been derived.

In [ ]:
changed = True                       # did this sweep add any new fact?

while changed:                       # keep sweeping until a pass adds nothing new
    changed = False
    for premises, conclusion in rules:
        already_known = conclusion in facts
        premises_hold = all(p in facts for p in premises)

        if not already_known and premises_hold:
            facts.add(conclusion)    # modus ponens fires: derive the conclusion
            print("derived", conclusion, "from", premises)
            changed = True

### Step 3 — Read off what got proven

When the loop stops, `facts` holds the full set of derivable symbols. Forward chaining is **sound** (it only adds things that genuinely follow) and, for definite clauses, **complete** (it finds everything that follows). So checking whether `E` is in `facts` is a complete test of whether `E` is entailed by our rules and starting facts.

In [ ]:
print("final known facts:", sorted(facts))
print("E proven?", "E" in facts)

## Visualize the data & results

_Starting from family facts about Tom, Bob, Ann and Pat, which new relationships does forward chaining derive?_

Same engine, richer facts. Now each fact is a **relation** like `parent(Tom, Bob)`, written as a tuple `("parent", "Tom", "Bob")`. The rules join facts together — exactly how a logic program (Prolog/Datalog) reasons about a family tree.

### Step 1 — State the family facts

We record what we directly observe: who is a parent of whom, and who is male. Each tuple is one ground fact. From just these we'll *derive* grandparent and grandfather relations that nobody stated explicitly.

In [ ]:
facts = {
    ("parent", "Tom", "Bob"),
    ("parent", "Bob", "Ann"),
    ("parent", "Bob", "Pat"),
    ("male", "Tom"),
    ("male", "Bob"),
}

order = []   # remember the sequence in which new facts get derived

print("given facts:", len(facts))

### Step 2 — Forward-chain two rules over the relations

Two rules drive the derivation:

- **grandparent(X, Z)** holds when `parent(X, Y)` and `parent(Y, Z)` share a middle person `Y` — we look for a pair of parent facts where the child of one is the parent of the other.
- **grandfather(X, Z)** holds when `grandparent(X, Z)` and `male(X)`.

As before, we sweep until no rule adds anything new. Each newly derived relation is appended to `order` so we can show the derivation sequence afterward.

In [ ]:
changed = True
while changed:
    changed = False

    # Rule: grandparent(X, Z) :- parent(X, Y), parent(Y, Z)
    for f1 in list(facts):
        if f1[0] == "parent":
            for f2 in list(facts):
                bridges = f2[0] == "parent" and f1[2] == f2[1]
                if bridges:
                    grandparent = ("grandparent", f1[1], f2[2])
                    if grandparent not in facts:
                        facts.add(grandparent)
                        order.append(grandparent)
                        changed = True

    # Rule: grandfather(X, Z) :- grandparent(X, Z), male(X)
    for f in list(facts):
        is_grandparent = f[0] == "grandparent"
        x_is_male = ("male", f[1]) in facts
        if is_grandparent and x_is_male:
            grandfather = ("grandfather", f[1], f[2])
            if grandfather not in facts:
                facts.add(grandfather)
                order.append(grandfather)
                changed = True

print("derived", order)

### Step 3 — Plot what was given vs. what was derived

The bar chart lines up the original parent facts against the four relations forward chaining produced: two grandparent links and the two grandfather links that follow once we add Tom's `male` fact. Blue is *given*; green and orange are *derived*.

In [ ]:
labels = ["parent facts", "gp(Tom,Ann)", "gp(Tom,Pat)", "gf(Tom,Ann)", "gf(Tom,Pat)"]
colors = ["#4ea1ff", "#7ee787", "#7ee787", "#ffb454", "#ffb454"]
tags   = ["given", "derived", "derived", "derived", "derived"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(labels, [1] * 5, color=colors)

for i, tag in enumerate(tags):
    ax.text(i, 1, tag, ha="center", va="bottom")

ax.set_ylim(0, 1.4)
ax.set_title("family facts before vs after forward chaining")
plt.show()